# REINFORCE with a learned baseline in NumPy

A hiker walks a trail with five forks and takes one of two paths at each. Some paths
pay at once and some only later, so even this small walk asks which choice earned which
reward. You will compute rewards-to-go, write the REINFORCE gradient of a softmax policy with
one baseline value per fork, check that its average over every possible walk is the exact
gradient, and train the policy with a baseline it learns as it goes.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# The trail: five forks, two paths at each. NEXT[s, a] is the fork that path a leads to from
# fork s (-1 when the walk ends there), and REWARD[s, a] is what the path pays: the views
# minus the effort, so a steep path can pay less now and more later.
NEXT = np.array([[1, 2], [3, 3], [4, 3], [-1, -1], [-1, -1]])
REWARD = np.array([[1.0, -1.0], [0.0, 1.0], [2.0, 0.0], [0.0, 1.0], [3.0, 0.0]])
N_FORKS, N_PATHS = REWARD.shape


def softmax_rows(theta):
    """Softmax over the last axis: each fork's row of logits gives that fork's policy."""
    shifted = theta - np.max(theta, axis=-1, keepdims=True)
    weights = np.exp(shifted)
    return weights / np.sum(weights, axis=-1, keepdims=True)


def sample_episode(theta, rng):
    """One walk under the policy softmax_rows(theta), as arrays (states, actions, rewards)."""
    states, actions, rewards = [], [], []
    s = 0
    while s != -1:
        a = int(rng.random() < softmax_rows(theta[s])[1])
        states.append(s)
        actions.append(a)
        rewards.append(REWARD[s, a])
        s = NEXT[s, a]
    return np.array(states), np.array(actions), np.array(rewards)


def state_values(theta):
    """The exact expected return from each fork onward under softmax_rows(theta)."""
    pi = softmax_rows(theta)
    V = np.zeros(N_FORKS)
    for s in reversed(range(N_FORKS)):          # every path leads to a higher-numbered fork
        for a in range(N_PATHS):
            later = V[NEXT[s, a]] if NEXT[s, a] >= 0 else 0.0
            V[s] += pi[s, a] * (REWARD[s, a] + later)
    return V


def expected_return(theta):
    """J(theta): the exact expected return of a walk from fork 0."""
    return state_values(theta)[0]


print(f"uniform policy: J = {expected_return(np.zeros((N_FORKS, N_PATHS))):.2f}")

## Part 1 (coding task): Rewards-to-go

Write `rewards_to_go(rewards, gamma)`. For one episode with rewards $r_0, \dots, r_{T-1}$ it
returns the float array of

$$
G_t = \sum_{k=t}^{T-1} \gamma^{\,k-t}\, r_k , \qquad t = 0, \dots, T-1 ,
$$

and leaves its input unchanged.

In [ ]:
def rewards_to_go(rewards, gamma=1.0):
    """G_t for every step t of one episode, as a float array."""
    # TODO
    raise NotImplementedError

In [ ]:
def same(a, b):
    return np.shape(a) == np.shape(b) and np.allclose(a, b, atol=1e-12)

r = [1, 0, 2]
G = rewards_to_go(r, 0.5)
assert not same(G, [1.0, 1.0, 3.0]), "these are running sums from the start of the episode: G_t adds the rewards from step t to the end"
assert not same(G, [1.5, 0.5, 0.5]), "each reward is discounted by gamma**k, its distance from the start: discount by its distance from step t"
assert same(G, [1.5, 1.0, 2.0]), f"rewards [1, 0, 2] with gamma 0.5 gave {G}, expected [1.5, 1.0, 2.0]"
assert r == [1, 0, 2], "rewards_to_go changed its input"
assert same(rewards_to_go(np.array([2.0, -1.0, 4.0]), 1.0), [5.0, 3.0, 4.0]), "wrong without discounting"
assert same(rewards_to_go(np.array([4.0]), 0.9), [4.0]), "a one-step episode has G_0 = r_0"
assert np.asarray(rewards_to_go(np.array([1, 2]), 1.0)).dtype.kind == "f", "return floats, even for integer rewards"
print("Part 1 passed")

## Part 2 (coding task): The gradient estimate

The policy has one row of logits per fork, `theta[s]`, and takes path `a` at fork `s` with
probability `softmax_rows(theta)[s, a]`. Write `policy_gradient(theta, episodes, baseline)`.
Each episode is a tuple `(states, actions, rewards)` like the ones `sample_episode` returns,
and `baseline` holds one number per fork. With no discounting, return the average over the
episodes of

$$
\sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t)\, \big(G_t - b(s_t)\big)
$$

as an array shaped like `theta`.

The check runs your function on two walks. Then it runs it on each of the eight walks the
trail allows, weights each result by the walk's probability, and compares that average with
the exact gradient of $J(\theta)$ from central differences.

In [ ]:
def policy_gradient(theta, episodes, baseline):
    """Average over the episodes of sum_t grad log pi(a_t | s_t) * (G_t - baseline[s_t])."""
    # TODO
    raise NotImplementedError

In [ ]:
theta_c = np.log(np.array([[3.0, 1.0], [1.0, 1.0], [1.0, 1.0], [1.0, 1.0], [4.0, 1.0]]))
walks = [
    (np.array([0, 2, 4]), np.array([1, 0, 0]), np.array([-1.0, 2.0, 3.0])),
    (np.array([0, 1, 3]), np.array([0, 1, 0]), np.array([1.0, 1.0, 0.0])),
]
b_c = np.array([3.0, 1.0, 4.0, 0.5, 2.5])
g = np.asarray(policy_gradient(theta_c, walks, b_c), dtype=float)
expected = np.array([[-0.5, 0.5], [0.0, 0.0], [0.25, -0.25], [-0.125, 0.125], [0.05, -0.05]])
assert g.shape == theta_c.shape, f"the gradient has shape {g.shape}, expected {theta_c.shape}"
assert not np.allclose(g, 2 * expected), "this is the sum over the two walks: return their average"
assert not np.allclose(g, -expected), "the sign is flipped: a path with a weight above zero should become more likely"
full_return = np.array([[-0.5, 0.5], [-0.25, 0.25], [0.0, 0.0], [0.375, -0.375], [0.15, -0.15]])
assert not np.allclose(g, full_return), "every step is weighted by its walk's whole return: weight step t by G_t, the rewards from t on"
if not np.allclose(g[0], expected[0]) and np.allclose(g[1:], expected[1:]):
    raise AssertionError("only fork 0 is wrong, the one fork both walks pass: add both walks' terms to its row (with fancy indexing, grad[rows] += values adds once per distinct row, while np.add.at adds every one)")
assert np.allclose(g, expected), f"on two hand-made walks the gradient is\n{np.round(g, 4)}\nexpected\n{expected}"


def all_walks(theta):
    """Every walk the trail allows, with its probability under softmax_rows(theta)."""
    pi = softmax_rows(theta)
    out = []

    def walk(s, p, states, actions, rewards):
        if s == -1:
            out.append((p, (np.array(states), np.array(actions), np.array(rewards))))
            return
        for a in range(N_PATHS):
            walk(NEXT[s, a], p * pi[s, a], states + [s], actions + [a], rewards + [REWARD[s, a]])

    walk(0, 1.0, [], [], [])
    return out


def numeric_gradient(theta, h=1e-6):
    G = np.zeros_like(theta)
    for idx in np.ndindex(theta.shape):
        up, down = theta.copy(), theta.copy()
        up[idx] += h
        down[idx] -= h
        G[idx] = (expected_return(up) - expected_return(down)) / (2 * h)
    return G


theta_r = np.random.default_rng(7).normal(0.0, 0.8, size=(N_FORKS, N_PATHS))
exact = numeric_gradient(theta_r)
for b in (np.zeros(N_FORKS), np.array([2.0, -1.0, 0.5, 3.0, 1.0])):
    average = sum(p * np.asarray(policy_gradient(theta_r, [w], b), dtype=float) for p, w in all_walks(theta_r))
    assert np.allclose(average, exact, atol=1e-6), f"averaged over all eight walks with baseline {b}, the estimate is\n{np.round(average, 4)}\nbut the exact gradient is\n{np.round(exact, 4)}"
print("Part 2 passed")

## Part 3 (coding task): Train with a learned baseline

Write `train(theta, iterations, batch_size, lr, baseline_lr, rng)`. Start from a copy of
`theta` and a baseline of zeros, one entry per fork. In each iteration:

1. Sample `batch_size` walks with `sample_episode(theta, rng)`.
2. Compute the gradient with `policy_gradient`, using the baseline as it stands before this batch.
3. Move the baseline toward what the batch saw: for every step of every walk,
   `b[s_t] += baseline_lr * (G_t - b[s_t])`.
4. Step uphill: `theta += lr * gradient`.

Return `(theta, baseline)`. The best walk earns 4. The check trains from the uniform policy and
compares the learned baseline with the exact values of the policy that training ends with.

In [ ]:
def train(theta, iterations, batch_size, lr, baseline_lr, rng):
    """REINFORCE with a per-fork baseline learned from the returns. Returns (theta, baseline)."""
    # TODO
    raise NotImplementedError

In [ ]:
theta0 = np.zeros((N_FORKS, N_PATHS))
theta_1, _ = train(theta0, iterations=1, batch_size=8, lr=0.5, baseline_lr=0.1, rng=np.random.default_rng(5))
assert not theta0.any(), "train changed the caller's theta: work on a copy"
rng_same = np.random.default_rng(5)
first_batch = [sample_episode(theta0, rng_same) for _ in range(8)]
first_step = 0.5 * np.asarray(policy_gradient(theta0, first_batch, np.zeros(N_FORKS)), dtype=float)
assert not np.allclose(theta_1, -first_step), "theta moved downhill: step uphill, theta += lr * gradient"
assert np.allclose(theta_1, first_step), "after one iteration from theta = 0, theta should be lr times the gradient of the first batch with the baseline still at zero: sample the walks with sample_episode(theta, rng), and compute the gradient before moving the baseline"

theta_t, b_t = train(theta0, iterations=300, batch_size=16, lr=0.5, baseline_lr=0.1, rng=np.random.default_rng(0))
theta_t, b_t = np.asarray(theta_t, dtype=float), np.asarray(b_t, dtype=float)
J0, J = expected_return(theta0), expected_return(theta_t)
V = state_values(theta_t)
print(f"J: {J0:.3f} -> {J:.3f}   (the best walk earns 4)")
print("learned baseline", np.round(b_t, 2))
print("exact values    ", np.round(V, 2))
assert J > J0, "the expected return went down: step uphill, theta += lr * gradient"
assert J > 3.9, f"J = {J:.3f} after 300 iterations, expected above 3.9"
assert np.any(b_t != 0), "the baseline never moved from zero"
on_path = [0, 2, 4]
assert np.allclose(b_t[on_path], V[on_path], atol=0.15), "on forks 0, 2 and 4, which the trained policy walks through, the baseline should be close to their exact values: move b[s_t] toward G_t, the return from step t on"
print("Part 3 passed")

## Solution

### Part 1: Rewards-to-go

In [ ]:
def rewards_to_go(rewards, gamma=1.0):
    """G_t for every step t of one episode, as a float array."""
    rewards = np.asarray(rewards, dtype=float)
    G = np.zeros_like(rewards)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = rewards[t] + gamma * running
        G[t] = running
    return G

**Why it works.** Going backward, each $G_t$ is $r_t + \gamma G_{t+1}$, built from the value
just computed, so one pass over the episode is enough. The two wrong arrays the check rejects
are the forward running sum, which adds up the past instead of the future, and discounting by
$\gamma^k$, which measures each reward's distance from the start of the episode rather than
from step $t$.

### Part 2: The gradient estimate

In [ ]:
def policy_gradient(theta, episodes, baseline):
    """Average over the episodes of sum_t grad log pi(a_t | s_t) * (G_t - baseline[s_t])."""
    pi = softmax_rows(np.asarray(theta, dtype=float))
    grad = np.zeros(pi.shape)
    for states, actions, rewards in episodes:
        G = rewards_to_go(rewards, 1.0)
        for s, a, g in zip(states, actions, G):
            score = -pi[s]
            score[a] += 1.0                     # one-hot(a) minus the policy at fork s
            grad[s] += score * (g - baseline[s])
    return grad / len(episodes)

**Why it works.** At a softmax row, $\partial \log \pi_\theta(a \mid s) / \partial \theta[s, b] =
\mathbb{1}[a = b] - \pi_\theta(b \mid s)$, and the rows of other forks get zero, so each step
adds one-hot minus the policy, times its weight, to its own fork's row. Adding inside a loop
keeps both walks' terms on fork 0, where a vectorized `grad[rows] += values` would keep only
one. The second check is the lesson's argument made exact: averaged over every walk with its
probability, the estimate equals the gradient of $J$, with no baseline and with a baseline of
arbitrary numbers, because each baseline value depends only on the fork.

### Part 3: Train with a learned baseline

In [ ]:
def train(theta, iterations, batch_size, lr, baseline_lr, rng):
    """REINFORCE with a per-fork baseline learned from the returns. Returns (theta, baseline)."""
    theta = np.array(theta, dtype=float)
    baseline = np.zeros(theta.shape[0])
    for _ in range(iterations):
        batch = [sample_episode(theta, rng) for _ in range(batch_size)]
        grad = policy_gradient(theta, batch, baseline)
        for states, _, rewards in batch:
            for s, g in zip(states, rewards_to_go(rewards, 1.0)):
                baseline[s] += baseline_lr * (g - baseline[s])
        theta += lr * grad
    return theta, baseline

**Why it works.** Each iteration estimates the gradient from fresh walks of the current
policy, as an on-policy method must. The baseline tracks the average return from each fork,
so the weight $G_t - b(s_t)$ says whether a walk did better or worse than usual from there,
and the steep first path, which pays $-1$ at once, is credited with the 2 and 3 that follow
it. The gradient uses the baseline from before the batch: updated first, $b(s_t)$ would
already contain this step's own return, which depends on the action it weights. By the end
the baseline is close to the exact values on the forks the policy walks, about 4, 5 and 3
on forks 0, 2 and 4, and lags behind on the forks it has stopped visiting.